In [1]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from shutil import copy2
from ultralytics import YOLO

In [2]:
cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "Training").exists():
    PROJECT_ROOT = cwd
elif cwd.name.lower() == "training" and (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd.parent

DATA_YAML = PROJECT_ROOT / "data" / "Football card.v2i.yolov8" / "data.yaml"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data file: {DATA_YAML}")
print(f"Models dir: {MODELS_DIR}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset yaml not found: {DATA_YAML}")

Project root: C:\Users\20102\Downloads\VAMOS
Data file: C:\Users\20102\Downloads\VAMOS\data\Football card.v2i.yolov8\data.yaml
Models dir: C:\Users\20102\Downloads\VAMOS\models


In [3]:
model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(PROJECT_ROOT / "runs"),
    name="football_cards_yolov8",
    exist_ok=True
)

print(train_results.save_dir)

New https://pypi.org/project/ultralytics/8.4.36 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\20102\Downloads\VAMOS\data\Football card.v2i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, m

In [5]:
best_weights = Path(train_results.save_dir) / "weights" / "cards.pt"
if not best_weights.exists():
    alt_best = Path(train_results.save_dir) / "weights" / "best.pt"
    alt_last = Path(train_results.save_dir) / "weights" / "last.pt"

    if alt_best.exists():
        best_weights = alt_best
    elif alt_last.exists():
        best_weights = alt_last
    else:
        raise FileNotFoundError(
            f"Best checkpoint not found. Checked: {best_weights}, {alt_best}, {alt_last}"
        )

best_model = YOLO(str(best_weights))

val_metrics = best_model.val(data=str(DATA_YAML), split="val")
test_metrics = best_model.val(data=str(DATA_YAML), split="test")

print("Validation metrics:", val_metrics.results_dict)
print("Test metrics:", test_metrics.results_dict)

Ultralytics 8.4.14  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 163.135.5 MB/s, size: 33.6 KB)
val: Scanning C:\Users\20102\Downloads\VAMOS\data\Football card.v2i.yolov8\valid\labels.cache... 20 images, 11 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 6.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.7s/it 5.3s<16.4s
                   all         20          9      0.811      0.741      0.752      0.491
              Red card          2          2      0.623        0.5       0.51      0.408
           Yellow card          7          7          1      0.982      0.995      0.574
Speed: 15.0ms preprocess, 26.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to C:\Users\20102\Downloads\VAMOS\Training\runs\detect\val
Ultralyt

In [6]:
destination = MODELS_DIR / "cards.pt"
copy2(best_weights, destination)
print(f"Saved best model to: {destination}")

Saved best model to: C:\Users\20102\Downloads\VAMOS\models\cards.pt
